In [1]:
# Importing the libraries
import pandas as pd
import numpy as np
from utils.fake_na_detection_and_cleaning import FakeNullDetector
from utils.sql_connector import SQLConnector
from utils.data_type_converter import DataTypeConverter
from utils.normalizer import Normalizer
from utils.mapping_categorical import MappingCategorical

In [2]:
# loading of 2023 datasets and object creations
db=SQLConnector('Stack_Overflow_Survey')
db.connect()
query='select * from Bronze.Survey_2023'
Survey_2023_df = db.read_query(query)

FakeNullDetector_obj = FakeNullDetector()
DataTypeConverter_obj = DataTypeConverter()
Normalizer_obj = Normalizer()
MappingCategorical_obj = MappingCategorical()

Successfully Connected to Stack_Overflow_Survey
Query executed successfully


c:\Users\Ayush\Git Repo\Stack-Over-Flow-Survey-Data-Engineering-Project\Data Warehouse\Silver Layer\utils\sql_connector.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [3]:
# Null detector and remover 
Survey_2023_df_cleaned=Survey_2023_df.copy()
FakeNullDetector_obj.detect_fake_nulls(Survey_2023_df_cleaned)
FakeNullDetector_obj.replace_fake_nulls(Survey_2023_df_cleaned)

{'AIAcc': {'NA': np.int64(50590)},
 'AIBen': {'NA': np.int64(27788)},
 'AIDevHaveWorkedWith': {'NA': np.int64(63280)},
 'AIDevWantToWorkWith': {'NA': np.int64(69597)},
 'AINextNeither different nor similar': {'NA': np.int64(53735)},
 'AINextSomewhat different': {'NA': np.int64(53735)},
 'AINextSomewhat similar': {'NA': np.int64(53735)},
 'AINextVery different': {'NA': np.int64(53735)},
 'AINextVery similar': {'NA': np.int64(53735)},
 'AISearchHaveWorkedWith': {'NA': np.int64(32856)},
 'AISearchWantToWorkWith': {'NA': np.int64(43034)},
 'AISelect': {'NA': np.int64(1211)},
 'AISent': {'NA': np.int64(27683)},
 'AIToolCurrently Using': {'NA': np.int64(51472)},
 'AIToolInterested in Using': {'NA': np.int64(51472)},
 'AIToolNot interested in Using': {'NA': np.int64(51472)},
 'BuyNewTool': {'NA': np.int64(6175)},
 'CodingActivities': {'NA': np.int64(15420)},
 'CompTotal': {'NA': np.int64(40959)},
 'ConvertedCompYearly': {'NA': np.int64(41165)},
 'Country': {'NA': np.int64(1211)},
 'Currency':

In [4]:
Survey_2023_df_cleaned['Employment'].value_counts()

In [5]:
# Mormalization of categorical columns : Basic mapping of values to reduce the number of unique values in each column and make it more consistent for analysis and visualization.
un_normalized_cols_name = ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase', 'MainBranch']
normalized_cols_name = ['Employment', 'Education_Level', 'Age', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'Current_Profession']
columns_map = [
    MappingCategorical_obj.get_map('employment_map'),
    MappingCategorical_obj.get_map('ed_level_map'),
    MappingCategorical_obj.get_map('age_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('operating_system_map'),
    MappingCategorical_obj.get_map('org_mapping'),
    MappingCategorical_obj.get_map('visit_freq_map'),
    MappingCategorical_obj.get_map('so_account_map'),
    MappingCategorical_obj.get_map('part_freq_map'),
    MappingCategorical_obj.get_map('comm_map'),
    MappingCategorical_obj.get_map('new_so_sites_map'),
    MappingCategorical_obj.get_map('survey_length_map'),
    MappingCategorical_obj.get_map('survey_ease_map'),
    MappingCategorical_obj.get_map('main_branch_map')
]
Survey_2023_df_cleaned = Normalizer_obj.normalize_categorical_columns_manual_mapping(Survey_2023_df_cleaned, un_normalized_cols_name, normalized_cols_name, columns_map)

[nan 'Employed, full-time'
 'Employed, full-time;Independent contractor, freelancer, or self-employed'
 'Not employed, but looking for work'
 'Independent contractor, freelancer, or self-employed'
 'Student, full-time'
 'Independent contractor, freelancer, or self-employed;Employed, part-time;Student, part-time'
 'Not employed, but looking for work;Student, full-time'
 'Employed, part-time;Student, part-time' 'Employed, part-time'
 'Student, full-time;Employed, part-time' 'I prefer not to say'
 'Employed, full-time;Independent contractor, freelancer, or self-employed;Employed, part-time'
 'Employed, full-time;Student, part-time'
 'Not employed, but looking for work;Employed, part-time'
 'Employed, full-time;Independent contractor, freelancer, or self-employed;Student, part-time'
 'Retired' 'Employed, full-time;Student, full-time'
 'Student, full-time;Student, part-time'
 'Independent contractor, freelancer, or self-employed;Student, full-time'
 'Not employed, and not looking for work' 

In [6]:
# Mormalization of categorical columns : changing NA to more meaningful values and make it more consistent for analysis and visualization.
nan_replacer_columns=[]
cleaned_nan_replacer_columns=[]
if nan_replacer_columns:
    Survey_2023_df_cleaned=Normalizer_obj.normalize_na_replacer_columns(Survey_2023_df_cleaned,nan_replacer_columns, cleaned_nan_replacer_columns)

In [7]:
# Mormalization of categorical columns : Multi-select columns where respondents could select multiple options, resulting in semicolon-separated values.
multi_select_cols = []
multi_select_normalized = []
multi_select_maps = [

]
if multi_select_cols:
    Normalizer_obj.normalize_categorical_columns_non_exploding(Survey_2023_df_cleaned, multi_select_cols, multi_select_normalized, multi_select_maps)

In [8]:
# Mormalization of categorical columns : we will create a mapping to group similar roles and responses together, reducing the number of unique values while preserving the overall meaning.
tech_stack_cols = ['LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'PlatformHaveWorkedWith', 'PlatformWantToWorkWith', 'WebframeHaveWorkedWith', 'WebframeWantToWorkWith', 'MiscTechHaveWorkedWith', 'MiscTechWantToWorkWith', 'ToolsTechHaveWorkedWith', 'ToolsTechWantToWorkWith', 'NEWCollabToolsHaveWorkedWith', 'NEWCollabToolsWantToWorkWith', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OfficeStackSyncHaveWorkedWith', 'OfficeStackSyncWantToWorkWith', 'AIDevHaveWorkedWith', 'AIDevWantToWorkWith', 'AISearchHaveWorkedWith', 'AISearchWantToWorkWith']
manual_mapping_cols = ['DevType', 'LearnCode']
all_target_cols = manual_mapping_cols + tech_stack_cols
manual_maps = {
    'DevType': MappingCategorical_obj.get_map('dev_type_map'),
    'LearnCode': MappingCategorical_obj.get_map('learn_code_map')
}
bridge_results = {}
target_names = [col + '_Clean' for col in all_target_cols]
maps_to_use = [manual_maps.get(col) for col in all_target_cols]
bridge_results = Normalizer_obj.normalize_categorical_columns_exploding(
    Survey_2023_df_cleaned, 
    all_target_cols, 
    target_names, 
    maps_to_use
)

--- Distribution for DevType_Clean ---
DevType_Clean
Full-stack             25735
Back-end               13745
Other/Unknown          12312
Front-end               5071
Desktop/Enterprise      3904
Other                   3615
DevOps                  2749
Researcher              2707
Mobile                  2597
Embedded/IoT            2131
Engineering Manager     2033
Student                 1996
Data Scientist/ML       1588
Executive               1332
Data Engineer           1248
Product Manager         1035
SRE                      901
Game/Graphics            866
Data/BI Analyst          837
SysAdmin                 743
QA/Testing               586
Educator                 415
Scientist                351
Designer                 281
DBA                      257
Marketing/Sales          149
Name: count, dtype: int64
------------------------------
--- Distribution for LearnCode_Clean ---
LearnCode_Clean
Other/Unknown            163135
Physical Media            45406
Online Certific

In [9]:
# Cleaning the year Columns
experience_cols = ['YearsCode', 'YearsCodePro']
Survey_2023_df_cleaned = Normalizer_obj.clean_years_columns(Survey_2023_df_cleaned, experience_cols)

In [10]:
# Cleaning the Currency column and Salary Columns
col = 'Currency'
Survey_2023_df_cleaned[col] = Survey_2023_df_cleaned[col].str.split().str[0]
nan_replacer_cols = ['Currency']
cleaned_nan_cols = ['Currency_Code']
Survey_2023_df_cleaned = Normalizer_obj.normalize_na_replacer_columns(
    Survey_2023_df_cleaned,
    nan_replacer_cols, 
    cleaned_nan_cols,
    replacer_value="Not Available"
)
numeric_target_cols = ['CompTotal', 'ConvertedCompYearly']
Survey_2023_df_cleaned = DataTypeConverter_obj.string_to_numeric(Survey_2023_df_cleaned, numeric_target_cols)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2023_df_cleaned, 'ConvertedCompYearly', 0.01, 0.95)
Normalizer_obj.fill_na_and_remove_outlier_percentile_method(Survey_2023_df_cleaned, 'CompTotal', 0.01, 0.95)

In [11]:
# Dropping the unrequired columns
# 1. Original columns that now have "Clean" versions
raw_redundant_cols = [col for col in ['Employment', 'EdLevel', 'Age', 'OpSysPersonal use', 'OpSysProfessional use', 'OrgSize', 'SOVisitFreq', 'SOAccount', 'SOPartFreq', 'SOComm', 'NEWSOSites', 'SurveyLength', 'SurveyEase', 'MainBranch'] if col != 'MainBranch'] + [] + [] + ['Currency']
# 2. Raw multi-select strings (already exploded into bridge_results)
multi_select_strings = all_target_cols
total_drop_list = list(set(raw_redundant_cols + multi_select_strings))
Survey_2023_df_cleaned.drop(columns=total_drop_list, inplace=True, errors='ignore')
print(f"Final Column Count: {len(Survey_2023_df_cleaned.columns)}")
print(Survey_2023_df_cleaned.columns.tolist())

Final Column Count: 60
['ResponseId', 'Q120', 'MainBranch', 'RemoteWork', 'CodingActivities', 'LearnCodeOnline', 'LearnCodeCoursesCert', 'YearsCode', 'YearsCodePro', 'PurchaseInfluence', 'TechList', 'BuyNewTool', 'Country', 'CompTotal', 'SOAI', 'AISelect', 'AISent', 'AIAcc', 'AIBen', 'AIToolInterested in Using', 'AIToolCurrently Using', 'AIToolNot interested in Using', 'AINextVery different', 'AINextNeither different nor similar', 'AINextSomewhat similar', 'AINextVery similar', 'AINextSomewhat different', 'TBranch', 'ICorPM', 'WorkExp', 'Knowledge_1', 'Knowledge_2', 'Knowledge_3', 'Knowledge_4', 'Knowledge_5', 'Knowledge_6', 'Knowledge_7', 'Knowledge_8', 'Frequency_1', 'Frequency_2', 'Frequency_3', 'TimeSearching', 'TimeAnswering', 'ProfessionalTech', 'Industry', 'ConvertedCompYearly', 'SurveyYear', 'Education_Level', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participa

In [12]:
# Data Type Conversion
numeric_cols = ['YearsCode', 'YearsCodePro', 'CompTotal', 'ConvertedCompYearly', 'SurveyYear']
categorical_cols = ['MainBranch', 'Country', 'Education_Level', 'OperatingSystem_Personal', 'OperatingSystem_Professional', 'Organization_Size', 'StackOverflow_Visit_Frequency', 'StackOverflow_Account_exists', 'StackOverflow_Participation_Frequency', 'StackOverflow_Community_Experience', 'NewStackOverflow_Sites', 'Survey_Length', 'Survey_Ease', 'Current_Profession', 'Currency_Code']
DataTypeConverter_obj.string_to_category(Survey_2023_df_cleaned, categorical_cols)
DataTypeConverter_obj.string_to_numeric(Survey_2023_df_cleaned, numeric_cols)
Survey_2023_df_cleaned['SurveyYear'] = Survey_2023_df_cleaned['SurveyYear'].fillna(0).astype('datetime64[ns]')
print(Survey_2023_df_cleaned.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89184 entries, 0 to 89183
Data columns (total 60 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   ResponseId                             89184 non-null  object        
 1   Q120                                   89184 non-null  object        
 2   MainBranch                             89184 non-null  category      
 3   RemoteWork                             73810 non-null  object        
 4   CodingActivities                       73764 non-null  object        
 5   LearnCodeOnline                        70084 non-null  object        
 6   LearnCodeCoursesCert                   37076 non-null  object        
 7   YearsCode                              89184 non-null  int64         
 8   YearsCodePro                           89184 non-null  int64         
 9   PurchaseInfluence                      64964 non-null  object

In [13]:
for table_name, bridge_df in bridge_results.items():
    value_col = [col for col in bridge_df.columns if col != 'ResponseId'][0]
    bridge_results[table_name][value_col] = bridge_df[value_col].astype('category')
    bridge_results[table_name] = DataTypeConverter_obj.string_to_numeric(bridge_results[table_name], ['ResponseId'])
    print(f"Fixed types for {table_name}: {bridge_results[table_name][value_col].dtype}")

Fixed types for DevType_Clean: category
Fixed types for LearnCode_Clean: category
Fixed types for LanguageHaveWorkedWith_Clean: category
Fixed types for LanguageWantToWorkWith_Clean: category
Fixed types for DatabaseHaveWorkedWith_Clean: category
Fixed types for DatabaseWantToWorkWith_Clean: category
Fixed types for PlatformHaveWorkedWith_Clean: category
Fixed types for PlatformWantToWorkWith_Clean: category
Fixed types for WebframeHaveWorkedWith_Clean: category
Fixed types for WebframeWantToWorkWith_Clean: category
Fixed types for MiscTechHaveWorkedWith_Clean: category
Fixed types for MiscTechWantToWorkWith_Clean: category
Fixed types for ToolsTechHaveWorkedWith_Clean: category
Fixed types for ToolsTechWantToWorkWith_Clean: category
Fixed types for NEWCollabToolsHaveWorkedWith_Clean: category
Fixed types for NEWCollabToolsWantToWorkWith_Clean: category
Fixed types for OfficeStackAsyncHaveWorkedWith_Clean: category
Fixed types for OfficeStackAsyncWantToWorkWith_Clean: category
Fixed ty

In [14]:
# Writing back to SQL
# Central Fact Table 2023
db.write_to_sql(df=Survey_2023_df_cleaned, schema='Silver', table_name='Survey_2023')
# Bridge Tables for Tech Stack and Manual Mapping Columns
for table_name, bridge_df in bridge_results.items():
    db.write_to_sql(df=bridge_df, schema='Silver', table_name=f"Bridge_{table_name}_2023")
db.close()

DataFrame written to Silver.Survey_2023 successfully.
DataFrame written to Silver.Bridge_DevType_Clean_2023 successfully.
DataFrame written to Silver.Bridge_LearnCode_Clean_2023 successfully.
DataFrame written to Silver.Bridge_LanguageHaveWorkedWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_LanguageWantToWorkWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_DatabaseHaveWorkedWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_DatabaseWantToWorkWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_PlatformHaveWorkedWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_PlatformWantToWorkWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_WebframeHaveWorkedWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_WebframeWantToWorkWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_MiscTechHaveWorkedWith_Clean_2023 successfully.
DataFrame written to Silver.Bridge_MiscTechWantToWorkWith_Cle

In [15]:
db.close()